# 00 — 環境健檢 + 認識這門課的「共用工具箱」

這門課的主角是 **LangGraph**，但如果完全沒碰過 LangChain 就直接跳進去，會看不懂「LangGraph 到底解決了什麼問題」。所以前兩份 notebook 會先帶你摸一下 LangChain 的基本寫法，之後每次看到 LangGraph 更簡潔的寫法，才知道它省掉了哪些麻煩。

## 這 15 份 notebook 的路線圖

可以把這門課想成一趟由淺入深的旅程：

```
[00 環境健檢] → [01 LangChain 基礎] → [02 三種寫法比較]
      → [03~09 LangGraph 核心與進階功能] → [10 部署] → [11 觀測]
      → [12 MCP] → [13 原生 SDK 對照] → [14 期末專案]
```

1. `00_setup_and_llm` — 環境健檢與共用 LLM 介面（這份）
2. `01_langchain_basics` — LCEL / Runnable / Chain
3. `02_three_agents_compare` — 三種寫法比較：手刻 while-loop vs `create_agent`（現成）vs 手刻 `StateGraph`
4. `03_langgraph_core` — StateGraph / State / Node / Edge
5. `04_langgraph_control_flow` — 條件邊、迴圈、`Command`
6. `05_langgraph_tools_and_agent` — ToolNode、bind_tools、手刻 ReAct loop
7. `06_langgraph_memory_checkpoint` — Checkpointer、thread_id、對話記憶
8. `07_langgraph_human_in_the_loop` — `interrupt()`、人工審核、resume
9. `08_langgraph_streaming` — `stream_mode` 種類與除錯
10. `09_langgraph_multi_agent` — Supervisor pattern、subgraph
11. `10_langgraph_persistence_deploy` — SqliteSaver/PostgresSaver、部署概念
12. `11_langsmith_observability` — LangSmith 追蹤與評估（選用，需要免費帳號）
13. `12_mcp_tools` — MCP 概念、本機 stdio MCP server、把工具接進 `ToolNode`
14. `13_provider_sdks` — 原生 OpenAI SDK / Anthropic (Claude) SDK 的工具呼叫格式對照
15. `14_capstone_it_ticket_agent` — 期末專案：IT 支援工單 agent，整合前面幾乎所有技巧

> 註：第 3 份原本規劃比較「舊版 `AgentExecutor`」、`create_agent`、`StateGraph` 三種寫法，但因為 `AgentExecutor` 在目前這個版本的 LangChain 已經被整個移除了，所以改成「自己手刻 while-loop」取代它的位置——細節那份 notebook 裡會解釋清楚。

## 為什麼環境要鎖 Python 3.12
系統預設的 Python 是 3.14，但 LangChain/LangGraph 依賴的套件（pydantic-core、tokenizers 等）在 3.14 上常常還沒有預先編譯好的版本，會退回「現場編譯原始碼」，很容易失敗。
所以這個 repo 用 `uv venv --python 3.12` 建一個獨立環境，並在 `pyproject.toml` 裡鎖住版本範圍——確保不管誰跑，結果都一致。

In [1]:
import sys

print(sys.version)
assert sys.version_info[:2] == (3, 12), "這個 repo 應該在 uv 建立的 3.12 venv 下執行"

3.12.13 (main, Jun  2 2026, 22:27:15) [Clang 22.1.3 ]


## 為什麼要「問套件」而不是「憑印象寫」

LangChain / LangGraph 這幾年 API 變動很大（`AgentExecutor` → `create_react_agent` → `langchain.agents.create_agent`，版本也已經進入 1.x）。如果照著舊教學或記憶裡的寫法直接下手，很容易踩到「這個 API 早就不存在了」的坑。

比喻一下：與其憑印象亂猜遙控器上第幾顆按鈕是什麼功能，不如直接翻說明書——這裡的「說明書」就是先把目前環境裝了哪個版本印出來看清楚。之後每份 notebook 用到新 API 前，也會先確認它真的存在（例如用 `dir()` 檢查一下），才動手寫，不憑印象瞎猜。

In [2]:
import importlib.metadata as metadata

for pkg in ["langchain", "langgraph", "langchain-openai", "langchain-core"]:
    print(f"{pkg}: {metadata.version(pkg)}")

langchain: 1.4.0
langgraph: 1.2.11
langchain-openai: 1.6.1
langchain-core: 1.6.2


## 統一的 LLM 介面：`get_llm()` —— 一支「萬用遙控器」

想像家裡 Sony 電視、Samsung 冷氣各有各的遙控器，很麻煩。如果有一支「萬用遙控器」，設定好品牌後，每個房間都用同一支操作——這就是 `get_llm()` 的角色。

- 不管背後接的是 OpenAI 還是其他家的模型，程式碼永遠寫同一行：`llm = get_llm()`
- 想換品牌／換模型？只改 `notebooks/_llm.py` 這一個檔案，其他教材程式碼完全不用動（`get_llm()` 內部用的是 LangChain 的 `init_chat_model()`——一個「不綁死特定廠牌」的統一入口）

**這門課完全不用付費 API key 也能上完。** 我們準備了一個「照劇本念台詞的演員」——`scripted_model()`。它長得跟真的模型一模一樣（可以 `.invoke()`、之後章節也能 `bind_tools()`），差別只是回答內容是事先寫好的劇本，不是真的想出來的。這讓每份教材都能離線、100% 重現地跑出結果，不會因為模型「今天心情不好」回答得不一樣。

如果你有付費的 `OPENAI_API_KEY`：把 `.env.example` 複製成 `.env`、填入 key，`get_llm()` 就會換成真的呼叫模型。

In [3]:
import sys

sys.path.insert(0, ".")  # 讓 notebooks/_llm.py 可以被 import

from _llm import get_llm, has_api_key

print("OPENAI_API_KEY 已設定:", has_api_key())

OPENAI_API_KEY 已設定: False


In [4]:
from _llm import scripted_model

offline_llm = scripted_model(["StateGraph 是一個用節點與邊描述執行流程的狀態機。"])
print(offline_llm.invoke("用一句話說明什麼是 LangGraph 的 StateGraph").content)

StateGraph 是一個用節點與邊描述執行流程的狀態機。


## 如果你有 API key：接真模型跑一次
把上面同一個問題丟給 `get_llm()`（真模型），你會看到模型真正想出來的答案，而不是照劇本念的台詞。沒有 key 也沒關係——上一格的 `scripted_model` 已經完整示範過整個流程了。

In [5]:
if has_api_key():
    llm = get_llm()
    response = llm.invoke("用一句話說明什麼是 LangGraph 的 StateGraph")
    print(response.content)
else:
    print("尚未設定 OPENAI_API_KEY，跳過真模型呼叫（上面 scripted_model 已經展示了完整流程）。")

尚未設定 OPENAI_API_KEY，跳過真模型呼叫（上面 scripted_model 已經展示了完整流程）。
